**STEP 0: Library Perequisites**

---



**STEP 1.1: Loading model directly using Ollama Library**

---



In [1]:
import ollama

print("Asking Ollama (with confidence scores)...")
response = ollama.chat(
    model='llama3:instruct',
    messages=[
        {'role': 'user', 'content': 'Who are you?'}
    ],
    options={
        'temperature': 0  
    },
    logprobs=True     
)

print(f"Answer: {response['message']['content']}\n")

if 'logprobs' in response:
    print("--- Confidence Scores (First 5 Tokens) ---")
    for token_data in response['logprobs'][:5]:
        token_str = token_data.get('token', '')
        score = token_data.get('logprob', -999) 
        
        # A score close to 0 (e.g., -0.01) is VERY confident.
        # A score like -2.5 is LOW confidence.
        print(f"Token: '{token_str}' | Logprob: {score:.4f}")
else:
    print("Logprobs not found. Ensure your Ollama server version is up to date.")

Asking Ollama (with confidence scores)...
Answer: I am LLaMA, an AI assistant developed by Meta AI that can understand and respond to human input in a conversational manner. I'm not a human, but a computer program designed to simulate conversation and answer questions to the best of my ability based on my training data.

My primary function is to provide information and assist with tasks through text-based conversations. I can generate human-like responses to a wide range of topics, including but not limited to:

1. General knowledge: History, science, technology, literature, and more.
2. Entertainment: Movies, TV shows, music, books, and games.
3. Education: Academic subjects like math, language, and social sciences.
4. Conversational dialogue: I can engage in natural-sounding conversations, using context and understanding to respond to questions and statements.

I'm constantly learning and improving my responses based on the interactions I have with users like you. So, feel free to a

**STEP 1.2: Loading model directly using OpenAI library Wrapper**

---



In [10]:
from openai import OpenAI

# 1. Initialize the Client to hijack the local Ollama server
client = OpenAI(
    base_url='http://localhost:11434/v1',  # Points to your local machine
    api_key='ollama'                       # Required string, but ignored by Ollama
)

print("Asking Ollama (via OpenAI Wrapper for detailed logprobs)...")

try:
    response = client.chat.completions.create(
        model="llama3:instruct",
        messages=[
            {'role': 'user', 'content': 'Who are you?'}
        ],
        temperature=0,      # 0 = Deterministic (Scientific standard)
        logprobs=True,      # Turn on confidence scores
        top_logprobs=5      # <--- THE KEY FEATURE: Get the top 5 alternatives per token
    )

    answer_text = response.choices[0].message.content
    print(f"Answer: {answer_text}\n")

    # We inspect the first n tokens to see if the model hesitated.
    logprob_data = response.choices[0].logprobs.content
    
    if logprob_data:
        print("--- Problogs: Confidence & Alternatives ---")
        PROBLOGS_COUNT = 5 # How many tokens to inspect
        for i, token_data in enumerate(logprob_data[:PROBLOGS_COUNT]):
            token_str = token_data.token
            score = token_data.logprob
            
            print(f"\nToken {i+1}: '{token_str}' | Score: {score:.4f}")
            
            if token_data.top_logprobs:
                print("   Alternatives considered:")
                for alt in token_data.top_logprobs:
                    print(f"   - '{alt.token}': {alt.logprob:.4f}")
    else:
        print("No logprobs returned. Check if your Ollama version supports the v1 API.")

except Exception as e:
    print(f"Error: {e}")
    print("Ensure 'pip install openai' is run and Ollama is running.")

Asking Ollama (via OpenAI Wrapper for detailed logprobs)...
Answer: I am LLaMA, an AI assistant developed by Meta AI that can understand and respond to human input in a conversational manner. I'm not a human, but a computer program designed to simulate conversation and answer questions to the best of my ability based on my training data.

My primary function is to provide information and assist with tasks through text-based conversations. I can generate human-like responses to a wide range of topics, including but not limited to:

1. General knowledge: History, science, technology, literature, and more.
2. Entertainment: Movies, TV shows, music, books, and games.
3. Education: Academic subjects like math, language, and social sciences.
4. Conversational dialogue: I can engage in natural-sounding conversations, using context and understanding to respond to questions and statements.

I'm constantly learning and improving my responses based on the interactions I have with users like you. 

**STEP 2: Define a string variable that contains a Prolog database**

___

In [18]:
import janus_swi as janus

# response = ollama.chat(
#     model='llama3:instruct',
#     messages=[
#         {'role': 'user', 'content': """
# Create a Prolog database for the following problem: 
# Alice has m sisters and n brothers. How many sisters does any of his brothers have? 

# Instructions: 
# While you will only provide the Prolog database and nothing else, internally reason as to how you will construct the database. Capture implicit logical reasoning paths and create rules for them. For example, any of Alice's siblings are also siblings of each other.
# Again, only provide the Prolog database as a string as your output. Don't add any "Here's the Prolog database" or any other text. Just the Prolog clauses, rules, and queries.

# Example Output:
# sister(alice, mary).
# brother(bob, alice).
# """}
#     ],
#     options={
#         'temperature': 0  
#     },
#     logprobs=True     
# )

# answer_text = response['message']['content']

# response = client.chat.completions.create(
#         model="llama3:instruct",
#         messages=[
#             {'role': 'user', 'content': """
# Instructions: 
# You will only provide the Prolog database for the given problem below and nothing else. Internally reason as to how you will construct the database. Capture implicit logical reasoning paths and create rules for them. For example, any of Alice's siblings are also siblings of each other.
# Again, only provide the Prolog database as a string as your output. Just the Prolog clauses, rules, and queries.

# Here is the problem: 
# Alice has 7 sisters and 3 brothers. How many sisters does any of his brothers have?
# Any of Alice's siblings are also siblings of each other.
# Alice is one of the sisters.

# Example Output:
# sister(alice, mary).
# brother(bob, alice).
# """
#             }
#         ],
#         temperature=0,      # 0 = Deterministic (Scientific standard)
#         logprobs=True,      # Turn on confidence scores
#         top_logprobs=5      # <--- THE KEY FEATURE: Get the top 5 alternatives per token
#     )

# answer_text = response.choices[0].message.content

# answer_text = """
# female(alice). female(s1). female(s2). female(s3). female(s4). female(s5). female(s6). female(s7). male(b1). male(b2). male(b3).

# is_sibling_of_alice(s1). is_sibling_of_alice(s2). is_sibling_of_alice(s3). is_sibling_of_alice(s4). is_sibling_of_alice(s5). is_sibling_of_alice(s6). is_sibling_of_alice(s7). is_sibling_of_alice(b1). is_sibling_of_alice(b2). is_sibling_of_alice(b3).

# sibling(X, Y) :- (X = alice, is_sibling_of_alice(Y)); (Y = alice, is_sibling_of_alice(X)). sibling(X, Y) :- is_sibling_of_alice(X), is_sibling_of_alice(Y), X = Y.

# sister(X, Y) :- female(X), sibling(X, Y).

# solve :- aggregate_all(count, sister(_, b1), Count), format('Any of the brothers has ~w sisters.', [Count]).

# ?- solve.
# """

answer_text = """
sister(alice, _). % Alice has 7 sisters
sister(_, X), sister(_, Y) -> X \= Y. % Each of Alice's sisters are distinct
brother(X, alice) -> integer(X), X > 3. % Alice has 3 brothers and they are all different from each other
brother(alice, _). % Alice is one of the sisters
sibling(S1, S2) :- sister(S1, _), sister(S2, _), S1 \= S2.
sibling(B1, B2) :- brother(B1, _), brother(B2, _), B1 \= B2.
brother(X, Y) -> sibling(Y, X). % Any of Alice's siblings are also siblings of each other
num_sisters_of_brother(X) :- brother(X, alice), sister(S, _), \+ (S = alice), num_sisters_of_brother(X, S).
num_sisters_of_brother(alice, 7). % Base case: Alice has 7 sisters
"""

print(f"{answer_text}\n")

janus.consult(file="file", data=answer_text)

<>:69: SyntaxWarning: invalid escape sequence '\='
<>:69: SyntaxWarning: invalid escape sequence '\='
/var/folders/15/f8gxc_d56jgc0gnxdrdfkrxr0000gn/T/ipykernel_18425/2543425967.py:69: SyntaxWarning: invalid escape sequence '\='
  sister(_, X), sister(_, Y) -> X \= Y. % Each of Alice's sisters are distinct
ERROR: file:3:
ERROR:    No permission to modify static procedure `(->)/2'
ERROR:    Defined at /usr/local/Cellar/swi-prolog/9.2.9/lib/swipl/boot/init.pl:401
ERROR: file:4:
ERROR:    No permission to modify static procedure `(->)/2'
ERROR:    Defined at /usr/local/Cellar/swi-prolog/9.2.9/lib/swipl/boot/init.pl:401



sister(alice, _). % Alice has 7 sisters
sister(_, X), sister(_, Y) -> X \= Y. % Each of Alice's sisters are distinct
brother(X, alice) -> integer(X), X > 3. % Alice has 3 brothers and they are all different from each other
brother(alice, _). % Alice is one of the sisters
sibling(S1, S2) :- sister(S1, _), sister(S2, _), S1 \= S2.
sibling(B1, B2) :- brother(B1, _), brother(B2, _), B1 \= B2.
brother(X, Y) -> sibling(Y, X). % Any of Alice's siblings are also siblings of each other
num_sisters_of_brother(X) :- brother(X, alice), sister(S, _), \+ (S = alice), num_sisters_of_brother(X, S).
num_sisters_of_brother(alice, 7). % Base case: Alice has 7 sisters




ERROR: file:8:
ERROR:    No permission to modify static procedure `(->)/2'
ERROR:    Defined at /usr/local/Cellar/swi-prolog/9.2.9/lib/swipl/boot/init.pl:401
